# Retrieval v2 — Colab Pro GPU launcher

## Goal
Notebook này **chỉ là launcher/orchestrator**: mount Google Drive, chuẩn bị source + public dataset, gọi `competition.run_retrieval_v2`, rồi kiểm tra manifest/submission. Toàn bộ thuật toán nằm trong repository.

Luồng chạy gồm 9 stage: `validate-input → keyframes → index → neighbors → segments → text-index → dense-index → predict advanced → validate-submission`. Metric và ground truth không được chạy vì hiện chưa có nhãn. `Experiment.md` chỉ được append sau khi cả 9 stage pass.

## Setup
Bật **Runtime → Change runtime type → GPU** trước khi chạy. Sửa duy nhất cell cấu hình dưới đây. `SOURCE_MODE='drive'` dùng checkout chưa push trong Drive; đổi sang `'git'` sau khi branch đã được push.

In [ ]:
from pathlib import Path
from datetime import datetime

# --- Chỉ sửa block này ---
SOURCE_MODE = 'drive'  # 'drive' hoặc 'git'
GIT_REPO_URL = 'https://github.com/24122013/AIChallenge26_Multimodal_Agentic_Video_Retrieval_System.git'
GIT_BRANCH = 'codex/retrieval-leaderboard-v2'
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/AIChallenge26_Multimodal_Agentic_Video_Retrieval_System')
PUBLIC_DATA_SOURCE = DRIVE_REPO_PATH / 'data' / 'public'  # thư mục hoặc file .zip
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/AIChallenge26/retrieval_runs')
MODEL_CACHE_ROOT = Path('/content/drive/MyDrive/AIChallenge26/model_cache')
RUN_ID = 'retrieval-v2-' + datetime.now().strftime('%Y%m%d-%H%M%S')
START_AT = 'validate-input'
STOP_AFTER = 'validate-submission'
VLM_MODE = 'off'  # off | optional | required; off là cấu hình final ổn định
DRY_RUN = False

REPO_ROOT = Path('/content/retrieval_repo')
PUBLIC_ROOT = Path('/content/public_data')
RUN_ROOT = DRIVE_RUNS_ROOT / RUN_ID
EXPERIMENT_REPORT = DRIVE_RUNS_ROOT / 'Experiment.md'
print({'run_id': RUN_ID, 'run_root': str(RUN_ROOT), 'public_source': str(PUBLIC_DATA_SOURCE)})


In [ ]:
import os, shlex, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')

def run_command(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print('$', shlex.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, env=env, check=True)

run_command(['nvidia-smi'])
print('Python:', sys.version)


## Step 1 — Chuẩn bị source code
Git mode clone đúng branch. Drive mode copy source sang local SSD của Colab để import/test nhanh hơn, nhưng bỏ qua `.venv` và toàn bộ generated artifact. Thư mục `.git` vẫn được giữ để run manifest ghi đúng commit + dirty diff hash.

In [ ]:
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

if SOURCE_MODE == 'git':
    run_command(['git', 'clone', '--branch', GIT_BRANCH, '--single-branch', GIT_REPO_URL, REPO_ROOT])
elif SOURCE_MODE == 'drive':
    if not (DRIVE_REPO_PATH / '.git').exists():
        raise FileNotFoundError(f'Drive repo phải chứa .git: {DRIVE_REPO_PATH}')
    REPO_ROOT.mkdir(parents=True)
    exclusions = [
        '--exclude=.venv', '--exclude=__pycache__', '--exclude=.pytest_cache',
        '--exclude=data/public', '--exclude=data/model_cache',
        '--exclude=competition/work', '--exclude=competition/keyframes',
        '--exclude=competition/metadata', '--exclude=competition/embeddings',
        '--exclude=competition/indexes', '--exclude=competition/results',
        '--exclude=competition/runs', '--exclude=competition/evaluation',
    ]
    run_command(['rsync', '-a', *exclusions, f'{DRIVE_REPO_PATH}/', f'{REPO_ROOT}/'])
else:
    raise ValueError("SOURCE_MODE phải là 'drive' hoặc 'git'")

run_command(['git', 'status', '--short', '--branch'], cwd=REPO_ROOT)
required = [REPO_ROOT / 'competition' / 'run_retrieval_v2.py', REPO_ROOT / 'configs' / 'retrieval.yaml']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Source thiếu architecture v2: ' + ', '.join(missing))


## Step 2 — Cài dependency
Giữ PyTorch CUDA có sẵn của Colab nếu nó đã thỏa constraint; cài các package còn lại từ `requirements.txt`, sau đó kiểm tra CUDA và FFmpeg trước khi bắt đầu model inference.

In [ ]:
run_command(['apt-get', 'update', '-qq'])
run_command(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'rsync'])
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_ROOT / 'requirements.txt'])
run_command([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))"] )
run_command(['ffmpeg', '-version'])


## Step 3 — Copy public dataset vào local SSD
`PUBLIC_DATA_SOURCE` có thể là thư mục chứa trực tiếp ba CSV + `videos/` + `vkis/`, hoặc file ZIP có một thư mục bao ngoài. Dataset được copy/extract sang `/content/public_data`; artifact và model cache vẫn ghi vào Drive để resume được.

In [ ]:
import zipfile

if PUBLIC_ROOT.exists():
    shutil.rmtree(PUBLIC_ROOT)
if PUBLIC_DATA_SOURCE.is_dir():
    shutil.copytree(PUBLIC_DATA_SOURCE, PUBLIC_ROOT)
elif PUBLIC_DATA_SOURCE.is_file() and PUBLIC_DATA_SOURCE.suffix.lower() == '.zip':
    PUBLIC_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(PUBLIC_DATA_SOURCE) as archive:
        archive.extractall(PUBLIC_ROOT)
    children = [path for path in PUBLIC_ROOT.iterdir() if path.name != '__MACOSX']
    if not (PUBLIC_ROOT / 'corpus.csv').is_file() and len(children) == 1 and children[0].is_dir():
        nested = children[0]
        for item in list(nested.iterdir()):
            shutil.move(str(item), PUBLIC_ROOT / item.name)
        nested.rmdir()
else:
    raise FileNotFoundError(f'Không tìm thấy public dataset: {PUBLIC_DATA_SOURCE}')

required = ['corpus.csv', 'questions.csv', 'sample_submission.csv', 'videos', 'vkis']
missing = [name for name in required if not (PUBLIC_ROOT / name).exists()]
if missing:
    raise FileNotFoundError('Public dataset thiếu: ' + ', '.join(missing))
run_command([sys.executable, '-m', 'competition.pipeline', 'validate-input', '--public-root', PUBLIC_ROOT], cwd=REPO_ROOT)


## Step 4 — Chạy end-to-end
Runner tự resume Phase 3, fail-closed nếu code/dataset/offline config khác manifest cũ, giải phóng process/model giữa các stage, build cả coarse index lẫn dense safety index, và dùng advanced retrieval. Nếu Colab bị ngắt, giữ nguyên `RUN_ID`, đặt `START_AT` ở stage cần chạy lại rồi chạy lại notebook.

In [ ]:
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env.update({
    'PYTHONUTF8': '1',
    'TOKENIZERS_PARALLELISM': 'false',
    'HF_HOME': str(MODEL_CACHE_ROOT / 'huggingface'),
    'TORCH_HOME': str(MODEL_CACHE_ROOT / 'torch'),
    'YOLO_CONFIG_DIR': str(MODEL_CACHE_ROOT / 'ultralytics'),
})
command = [
    sys.executable, '-m', 'competition.run_retrieval_v2',
    '--public-root', PUBLIC_ROOT,
    '--run-root', RUN_ROOT,
    '--experiment-report', EXPERIMENT_REPORT,
    '--experiment-note', 'Colab Pro retrieval v2 E2E; ground truth/metrics unavailable',
    '--device', 'cuda', '--batch-size', 'auto', '--num-workers', '0',
    '--candidate-interval-sec', '0.5', '--max-gap-seconds', '2.0',
    '--target-density-per-second', '0.5',
    '--dedup-similarity-threshold', '0.92',
    '--endpoint-protection', 'on',
    '--model-cache-root', MODEL_CACHE_ROOT,
    '--coarse-top-n', '50', '--dense-global-top-k', '300',
    '--dense-rescue-clips', '10', '--max-candidate-clips', '60',
    '--dense-frames-per-clip', '12', '--rrf-k', '60',
    '--modality-hint-boost', '1.5', '--vlm-mode', VLM_MODE,
    '--start-at', START_AT, '--stop-after', STOP_AFTER,
]
if DRY_RUN:
    command.append('--dry-run')
run_command(command, cwd=REPO_ROOT, env=env)


## Checks
Cell này không đánh giá chất lượng retrieval. Nó chỉ xác nhận architecture contract: đủ 9 stage, dense rows khớp FAISS, submission 100×100 hợp lệ, không answer trùng, và checksum submission đã được bind vào run manifest.

In [ ]:
import hashlib, json

if DRY_RUN:
    print('Dry-run hoàn tất; chưa có artifact để validate.')
else:
    manifest_path = RUN_ROOT / 'run_manifest.json'
    submission_path = RUN_ROOT / 'results' / 'submission.csv'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected_stages = ['validate-input', 'keyframes', 'index', 'neighbors', 'segments', 'text-index', 'dense-index', 'predict', 'validate-submission']
    assert manifest['status'] == 'architecture_complete', manifest['status']
    assert all(manifest['stages'][stage]['status'] == 'passed' for stage in expected_stages)
    run_command([sys.executable, '-m', 'competition.pipeline', 'validate-dense-index', '--run-root', RUN_ROOT], cwd=REPO_ROOT)
    run_command([sys.executable, '-m', 'competition.pipeline', 'validate-submission', '--public-root', PUBLIC_ROOT, '--submission-path', submission_path], cwd=REPO_ROOT)
    submission_sha = hashlib.sha256(submission_path.read_bytes()).hexdigest()
    assert manifest['submission']['sha256'] == submission_sha
    print(json.dumps({
        'status': manifest['status'],
        'run_id': manifest['run_id'],
        'candidate_count': manifest['offline'].get('candidate_count'),
        'selected_count': manifest['offline'].get('selected_count'),
        'submission': str(submission_path),
        'submission_sha256': submission_sha,
        'experiment_report': str(EXPERIMENT_REPORT),
    }, indent=2))


## Next Steps
1. Nộp file `<RUN_ROOT>/results/submission.csv`.
2. Sau khi có điểm thật, bind điểm vào đúng checksum bằng `python -m competition.pipeline record-score --run-root <RUN_ROOT> --score <SCORE> --split public`.
3. Chỉ promote khi score vượt baseline bằng `python -m competition.pipeline promote-run --run-root <RUN_ROOT> --minimum-score 0.818`.
4. Không đổi config rồi resume cùng `RUN_ID`; hãy tạo run mới để tránh trộn lineage.